# Match metadata: diagnostics gallery

Read-only diagnostics for inspecting existing matching and metadata outputs.
This notebook should not run matching or assignment stages.


In [1]:
# Shared setup for diagnostics scripts.
import os
from pathlib import Path

import src.rainfall_rescue_sqlite as _pkg
from src.rainfall_rescue_sqlite.parquet_ingest import default_ensemble_parquet_root
from src.rainfall_rescue_sqlite.parquet_similarity import default_comparison_parquet_root

repo_root = Path(_pkg.__file__).resolve().parents[2]
ensemble_dataset_root = default_ensemble_parquet_root()

# Diagnostics always read the canonical main comparison dataset.
comparison_root = default_comparison_parquet_root()

repo_root, ensemble_dataset_root, comparison_root


(PosixPath('/home/users/philip.brohan/Projects/Auto-Daily-Rainfall-QC-MO'),
 PosixPath('/data/scratch/philip.brohan/ADRQ/ensemble_transcriptions_parquet'),
 PosixPath('/data/scratch/philip.brohan/ADRQ/monthly_similarity_parquet'))

In [2]:
# Dataset provenance and size summary for this diagnostics run.
#
# All diagnostics cells below should use these shared variables so they read
# exactly the same published session from the canonical metadata comparison dataset.

import duckdb
from src.rainfall_rescue_sqlite.run_manifest import (
    current_manifest_path,
    load_current_run_manifest,
)

# Canonical diagnostics root (set in setup cell).
diagnostics_comparison_root = Path(comparison_root).resolve()
diagnostics_ensemble_root = Path(ensemble_dataset_root)

manifest_path = current_manifest_path(diagnostics_comparison_root)
manifest = load_current_run_manifest(
    diagnostics_comparison_root,
    expected_pipeline="match_metadata",
    require_root_match=True,
)

# Manifest session_id refers to the published ensemble_metadata session.
diagnostics_session_id = int(manifest["session_id"])
diagnostics_metadata_file = Path(manifest["ensemble_metadata_path"]).resolve()
manifest_comparison_root = Path(manifest["comparison_root"]).resolve()

# Optional stage-input metadata files captured at publish time.
_data_metadata_input_path = manifest.get("data_metadata_input_path")
_allsheets_metadata_input_path = manifest.get("allsheets_metadata_input_path")
diagnostics_data_metadata_file = (
    Path(_data_metadata_input_path).resolve()
    if _data_metadata_input_path
    else None
)
diagnostics_allsheets_metadata_file = (
    Path(_allsheets_metadata_input_path).resolve()
    if _allsheets_metadata_input_path
    else None
)

if manifest_comparison_root != diagnostics_comparison_root:
    raise SystemExit(
        f"Manifest comparison_root mismatch: {manifest_comparison_root} != {diagnostics_comparison_root}"
    )

try:
    diagnostics_metadata_file.relative_to(diagnostics_comparison_root)
except ValueError as exc:
    raise SystemExit(
        f"Manifest metadata file is outside comparison root: {diagnostics_metadata_file}"
    ) from exc

# Stage-input files may live under different roots (e.g. ALLSHEETS residual root),
# so only require that they exist when paths are present in the manifest.
for _label, _path in (
    ("DATA metadata input", diagnostics_data_metadata_file),
    ("ALLSHEETS metadata input", diagnostics_allsheets_metadata_file),
):
    if _path is None:
        continue
    if not _path.exists():
        raise SystemExit(f"Manifest {_label} file does not exist: {_path}")

sim_sessions_glob = diagnostics_comparison_root / "similarity_sessions" / "*.parquet"
sim_matches_glob = diagnostics_comparison_root / "similarity_matches" / "*.parquet"
ensemble_files_glob = diagnostics_ensemble_root / "ensemble_files" / "*.parquet"
ensemble_daily_glob = diagnostics_ensemble_root / "ensemble_daily_values" / "*.parquet"

if not diagnostics_metadata_file.exists():
    raise SystemExit(
        f"Manifest metadata file does not exist: {diagnostics_metadata_file}"
    )
if not any((diagnostics_comparison_root / "similarity_sessions").glob("*.parquet")):
    raise SystemExit(
        f"No similarity_sessions parquet found under {diagnostics_comparison_root}."
    )

# Keep the metadata-file session id for reporting/consistency checks.
try:
    diagnostics_metadata_file_session_id = int(
        diagnostics_metadata_file.stem.split("_")[1]
    )
except (IndexError, ValueError):
    raise SystemExit(
        f"Unexpected metadata file name format: {diagnostics_metadata_file.name}"
    )

if diagnostics_metadata_file_session_id != diagnostics_session_id:
    raise SystemExit(
        f"Manifest session_id ({diagnostics_session_id}) does not match metadata file session "
        f"({diagnostics_metadata_file_session_id})."
    )

conn = duckdb.connect()
try:
    # Similarity sessions are independent from metadata-session numbering.
    diagnostics_similarity_session_id = conn.execute(
        f"SELECT MAX(session_id) FROM read_parquet('{sim_sessions_glob}')"
    ).fetchone()[0]
    if diagnostics_similarity_session_id is None:
        raise SystemExit(
            f"No similarity session_id found under {diagnostics_comparison_root}."
        )
    diagnostics_similarity_session_id = int(diagnostics_similarity_session_id)

    diagnostics_session_rows = conn.execute(
        f"""
        SELECT COUNT(*)
        FROM read_parquet('{sim_sessions_glob}')
        WHERE session_id = ?
        """
        , [diagnostics_similarity_session_id]
    ).fetchone()[0]
    if int(diagnostics_session_rows) <= 0:
        raise SystemExit(
            f"Similarity session_id {diagnostics_similarity_session_id} not found in similarity_sessions under {diagnostics_comparison_root}."
        )

    diagnostics_metadata_rows = conn.execute(
        f"SELECT COUNT(*) FROM read_parquet('{diagnostics_metadata_file}')"
    ).fetchone()[0]
    diagnostics_located_rows = conn.execute(
        f"""
        SELECT COUNT(*)
        FROM read_parquet('{diagnostics_metadata_file}')
        WHERE matched_latitude IS NOT NULL
          AND matched_longitude IS NOT NULL
        """
    ).fetchone()[0]

    diagnostics_similarity_rows = conn.execute(
        f"""
        SELECT COUNT(*)
        FROM read_parquet('{sim_matches_glob}')
        WHERE session_id = ?
        """
        , [diagnostics_similarity_session_id]
    ).fetchone()[0]
    diagnostics_rank1_rows = conn.execute(
        f"""
        SELECT COUNT(*)
        FROM read_parquet('{sim_matches_glob}')
        WHERE session_id = ? AND query_rank = 1
        """
        , [diagnostics_similarity_session_id]
    ).fetchone()[0]

    diagnostics_ensemble_files = conn.execute(
        f"SELECT COUNT(*) FROM read_parquet('{ensemble_files_glob}')"
    ).fetchone()[0]
    diagnostics_ensemble_daily_rows = conn.execute(
        f"SELECT COUNT(*) FROM read_parquet('{ensemble_daily_glob}')"
    ).fetchone()[0]

    diagnostics_data_stage_matched_rows = None
    if diagnostics_data_metadata_file is not None:
        diagnostics_data_stage_matched_rows = conn.execute(
            f"""
            SELECT COUNT(*)
            FROM read_parquet('{diagnostics_data_metadata_file}')
            WHERE match_type IS NOT NULL
            """
        ).fetchone()[0]

    diagnostics_allsheets_stage_matched_rows = None
    if diagnostics_allsheets_metadata_file is not None:
        diagnostics_allsheets_stage_matched_rows = conn.execute(
            f"""
            SELECT COUNT(*)
            FROM read_parquet('{diagnostics_allsheets_metadata_file}')
            WHERE match_type = 'exact'
            """
        ).fetchone()[0]

    diagnostics_allsheets_final_rows = conn.execute(
        f"""
        SELECT COUNT(*)
        FROM read_parquet('{diagnostics_metadata_file}')
        WHERE match_type = 'exact_allsheets'
        """
    ).fetchone()[0]
    diagnostics_allsheets_final_with_coords = conn.execute(
        f"""
        SELECT COUNT(*)
        FROM read_parquet('{diagnostics_metadata_file}')
        WHERE match_type = 'exact_allsheets'
          AND matched_latitude IS NOT NULL
          AND matched_longitude IS NOT NULL
        """
    ).fetchone()[0]
finally:
    conn.close()

print("Diagnostics data source")
print(f"  comparison root:         {diagnostics_comparison_root}")
print(f"  ensemble root:           {diagnostics_ensemble_root}")
print(f"  manifest:                {manifest_path}")
print(f"  manifest pipeline:       {manifest.get('pipeline')}")
print(f"  manifest published_at:   {manifest.get('published_at_utc')}")
print(f"  manifest git_commit:     {manifest.get('git_commit')}")
print(f"  metadata file:           {diagnostics_metadata_file.name}")
print(f"  metadata file session:   {diagnostics_metadata_file_session_id}")
print(f"  metadata session_id:     {diagnostics_session_id}")
print(f"  similarity session_id:   {diagnostics_similarity_session_id}")
print(f"  DATA metadata input:     {diagnostics_data_metadata_file.name if diagnostics_data_metadata_file is not None else 'n/a'}")
print(f"  ALLSHEETS metadata input:{diagnostics_allsheets_metadata_file.name if diagnostics_allsheets_metadata_file is not None else 'n/a'}")
print("Dataset size summary")
print(f"  ensemble_metadata rows (file):    {diagnostics_metadata_rows:,}")
print(f"  located metadata rows:            {diagnostics_located_rows:,}")
print(f"  similarity_matches rows (session):{diagnostics_similarity_rows:,}")
print(f"  rank-1 similarity rows:           {diagnostics_rank1_rows:,}")
print(f"  ensemble files:                   {diagnostics_ensemble_files:,}")
print(f"  ensemble daily rows:              {diagnostics_ensemble_daily_rows:,}")
print("Stage match summary")
if diagnostics_data_stage_matched_rows is None:
    print("  initial similarity (DATA):        n/a (manifest has no data_metadata_input_path)")
else:
    print(f"  initial similarity (DATA) matched rows:          {diagnostics_data_stage_matched_rows:,}")
if diagnostics_allsheets_stage_matched_rows is None:
    print("  ALLSHEETS matching stage:         n/a (manifest has no allsheets_metadata_input_path)")
else:
    print(f"  ALLSHEETS matching stage matched rows:           {diagnostics_allsheets_stage_matched_rows:,}")
print(f"  ALLSHEETS rows in final metadata:                {diagnostics_allsheets_final_rows:,}")
print(f"  ALLSHEETS rows with known lat/lon in final file: {diagnostics_allsheets_final_with_coords:,}")

Diagnostics data source
  comparison root:         /data/scratch/philip.brohan/ADRQ/monthly_similarity_parquet
  ensemble root:           /data/scratch/philip.brohan/ADRQ/ensemble_transcriptions_parquet
  manifest:                /data/scratch/philip.brohan/ADRQ/monthly_similarity_parquet/run_manifest/current.json
  manifest pipeline:       match_metadata
  manifest published_at:   2026-09-01T12:36:43+00:00
  manifest git_commit:     5119cb33fdb85b21352fb0fb86cd3ed2fc4643f0
  metadata file:           session_000002.parquet
  metadata file session:   2
  metadata session_id:     2
  similarity session_id:   1
  DATA metadata input:     session_000001.parquet
  ALLSHEETS metadata input:session_000001.parquet
Dataset size summary
  ensemble_metadata rows (file):    515,469
  located metadata rows:            446,133
  similarity_matches rows (session):5,038,830
  rank-1 similarity rows:           503,883
  ensemble files:                   584,513
  ensemble daily rows:              1,087

In [6]:
# Show unique ALLSHEETS station names, deduplicated by the matched station name
# itself. A station name is counted once in whichever bucket it appears in after
# resolving any duplicate matches to the same name; the printed counts therefore
# reflect the deduplicated name list, not the raw row count.

import duckdb

conn = duckdb.connect()
try:
    rows = conn.execute(
        f"""
        SELECT matched_location_name,
               matched_latitude IS NOT NULL AND matched_longitude IS NOT NULL AS has_location
        FROM read_parquet('{diagnostics_metadata_file}')
        WHERE match_type = 'exact_allsheets'
        ORDER BY matched_location_name NULLS LAST
        """
    ).fetchall()
finally:
    conn.close()

seen = {}
for station_name, has_location in rows:
    name = str(station_name).strip() if station_name not in (None, "") else "<missing station name>"
    key = name.casefold()
    if key not in seen:
        seen[key] = {"name": name, "has_location": bool(has_location)}
    else:
        seen[key]["has_location"] = seen[key]["has_location"] or bool(has_location)

with_location = sorted(
    item["name"] for item in seen.values() if item["has_location"]
)
without_location = sorted(
    item["name"] for item in seen.values() if not item["has_location"]
)

print(f"Unique ALLSHEETS exact_allsheets names: {len(seen)}")
print(f"  with location:    {len(with_location)}")
print(f"  without location: {len(without_location)}")
print()

print("ALLSHEETS station names WITH a resolved location:")
if with_location:
    for name in with_location:
        print(f"  {name}")
else:
    print("  (none)")
print()

print("ALLSHEETS station names WITHOUT a resolved location:")
if without_location:
    for name in without_location:
        print(f"  {name}")
else:
    print("  (none)")

from pathlib import Path
out_path = Path(repo_root) / "ALLSHEETS_used.txt"
out_lines = [
    "ALLSHEETS station names WITH a resolved location:",
    *with_location,
    "",
    "ALLSHEETS station names WITHOUT a resolved location:",
    *without_location,
]
out_path.write_text("\n".join(out_lines) + "\n", encoding="utf-8")
print(f"Saved deduplicated names to {out_path}")


Unique ALLSHEETS exact_allsheets names: 3607
  with location:    1746
  without location: 1861

ALLSHEETS station names WITH a resolved location:
  (EGGERSLACK )
  ABBOTSBURY (COTONEASTER)
  ABBOTSBURY MARKET STREET
  ABBOTSHAM HIGH PARK
  ABBOTSKERSWELL (THE MANOR HOUSE)
  ABERGELE (COUNTY SCHOOL)
  ABERGELE KINMEL PARK
  ABERLLEFENNI
  ABERYSTWYTH (PEN-Y-CWM)
  ABERYSTWYTH (PENGLAIS HILL)
  ABERYSTWYTH (TROED-RHIW-SEIRI) CARDIGAN
  ABERYSTWYTH (UNIV. FARM LOWER)
  ABERYSTWYTH UNIVERSITY COLLEGE
  ABINGDON (BATH STREET)
  ABINGDON DRAYCOTT MOOR,
  ABINGDON DRAYTON
  ABINGDON FITZHARRIS
  ABINGER HAMMER, WEST HACKHURST
  ACKWORTH FLOUNDERS INSTITUTE
  ACTON BURNELL
  ADISHAM RECTORY
  ALBURY
  ALBURY, WESTON LODGE
  ALDEBY VICARAGE
  ALDERLEY EDGE (ELM GROVE)
  ALDERMASTON (BEENHAM HILL)
  ALDERSHOT (GAS WORKS)
  ALDERSHOT GAS WORKS
  ALFORD VICARAGE
  ALFRISTON VICARAGE
  ALLHALLOWS AND STOKE SCHOOL
  ALMELEY (THE BUNGALOW)
  ALMELEY SCHOOL
  ALPHINGTON
  ALRESFORD (PERINS GRAMMAR SCH

## Diagnostic figure for a single transcription

`scripts/diagnostics/plot_image_consensus_metadata.py` builds a one-page
diagnostic for any daily-data specifier (the ensemble file-name stem, e.g.
`DRain_1911-1920_RainNos_Middlesex_H-P-17`). It pulls everything from the
project parquet datasets and shows, left to right:

- the **original scanned image**,
- the **daily transcription consensus** as a text table (median over the 5
  members; cells where the members disagree are drawn in red), with a monthly
  **Totals** row,
- the **monthly-total comparison** &mdash; all 5 ensemble member monthly values
  vs a selected-rank RR station-year (set by `comparison_rank`), plus a
  "member − RR rank-N" differences panel, and
- a **UK map** showing only the same selected-rank matched station location.

From the command line:

```bash
python scripts/diagnostics/plot_image_consensus_metadata.py \
    --specifier DRain_1911-1920_RainNos_Middlesex_H-P-17 \
    --comparison-rank 1 --top-k 5 --output diagnostic.webp
```

The cell below imports the script's `build_figure` and renders it inline.

In [ ]:
# Demonstrate the diagnostic plot script, rendered inline.

import importlib.util

from IPython.display import Image, display

script_path = repo_root / "scripts" / "diagnostics" / "plot_image_consensus_metadata.py"

spec = importlib.util.spec_from_file_location("diag_plot", script_path)
diag_plot = importlib.util.module_from_spec(spec)
spec.loader.exec_module(diag_plot)

specifier ="DRain_1871-1880_Bedfordshire-17"
comparison_rank = 1  # Change to 2, 3, ... to compare with another rank.
output_path = Path(
    f"{os.getenv('PDIR')}/diagnostics/{specifier}_rank{comparison_rank}_diagnostic.webp"
)

diag_plot.build_figure(
    specifier=specifier,
    ensemble_dataset_root=diagnostics_ensemble_root,
    comparison_root=diagnostics_comparison_root,
    top_k=5,
    comparison_rank=comparison_rank,
    output_path=output_path,
)

print(f"Comparison root: {diagnostics_comparison_root}")
print(f"Metadata file (locked): {diagnostics_metadata_file.name}")
print(f"Locked session_id: {diagnostics_session_id}")
print(f"Comparison rank: {comparison_rank}")
display(Image(filename=str(output_path)))


In [ ]:
# Plot exact-agreement-count distribution for any match rank, rendered inline.

import importlib.util

from IPython.display import Image, display

rank_dist_script_path = repo_root / "scripts" / "diagnostics" / "plot_rank1_exact_agreement_distribution.py"

spec_rank_dist = importlib.util.spec_from_file_location("rank_dist_plot", rank_dist_script_path)
rank_dist_plot = importlib.util.module_from_spec(spec_rank_dist)
spec_rank_dist.loader.exec_module(rank_dist_plot)

rank_to_plot = 1
rank_dist_output_path = Path(
    f"{os.getenv('PDIR')}/diagnostics/rank{rank_to_plot}_exact_agreement_distribution.webp"
)

# Use the similarity session id (from Cell 3), not the metadata session id.
session_id, counts, total = rank_dist_plot.load_distribution_parquet(
    diagnostics_comparison_root,
    session_id=int(diagnostics_similarity_session_id),
    rank=rank_to_plot,
 )
rank_dist_plot.plot_distribution(
    counts,
    total,
    session_id,
    rank_to_plot,
    rank_dist_output_path,
)

print(f"Comparison root: {diagnostics_comparison_root}")
print(f"Similarity session (locked): {session_id}")
print(f"Rank plotted: {rank_to_plot}")
print(f"Total rank-{rank_to_plot} rows: {total}")

display(Image(filename=str(rank_dist_output_path)))

## Map of consensus daily rainfall for a single date

`scripts/diagnostics/plot_daily_rainfall_map.py` takes a calendar date
`YYYY-MM-DD` and plots every located station's **consensus** daily rainfall (the
median over the 5 ensemble members) on a map of the UK. A station is included
when its ensemble file was assigned a Rainfall-Rescue year matching the date's
year (`matched_year`) and has an assigned latitude/longitude.

From the command line:

```bash
python scripts/diagnostics/plot_daily_rainfall_map.py 1931-10-15 \
    --output daily_map.webp
```

The cell below imports the script's `build_figure` and renders it inline.

In [ ]:
# Demonstrate the daily-rainfall map diagnostic, rendered inline.

import importlib.util
import os
from datetime import date
from pathlib import Path

import duckdb
from IPython.display import Image, display

daily_map_script_path = repo_root / "scripts" / "diagnostics" / "plot_daily_rainfall_map.py"

spec_daily_map = importlib.util.spec_from_file_location("daily_map_plot", daily_map_script_path)
daily_map_plot = importlib.util.module_from_spec(spec_daily_map)
spec_daily_map.loader.exec_module(daily_map_plot)

def _located_count_for_year(metadata_file: Path, target_year: int) -> int:
    conn = duckdb.connect()
    try:
        n = conn.execute(
            f"""
            SELECT COUNT(*)
            FROM read_parquet('{metadata_file}')
            WHERE matched_year = ?
              AND matched_latitude IS NOT NULL
              AND matched_longitude IS NOT NULL
            """
            , [target_year]
        ).fetchone()[0]
    finally:
        conn.close()
    return int(n)

target_date = date(1895, 11, 3)  # Change to any YYYY-MM-DD with matched records.
comparison_root_for_map = diagnostics_comparison_root
map_session_id = int(diagnostics_session_id)
located_n_for_year = _located_count_for_year(
    diagnostics_metadata_file, target_date.year
)
if located_n_for_year <= 0:
    raise SystemExit(
        f"No located ensemble records found for {target_date.isoformat()} "
        f"in locked metadata file {diagnostics_metadata_file.name}."
    )

daily_map_output_path = Path(
    f"{os.getenv('PDIR')}/diagnostics/daily_map_{target_date.isoformat()}.webp"
)

daily_map_plot.build_figure(
    target_date=target_date,
    ensemble_dataset_root=diagnostics_ensemble_root,
    comparison_root=comparison_root_for_map,
    session_id=map_session_id,
    output_path=daily_map_output_path,
)

print(f"Date: {target_date.isoformat()}")
print(f"Using comparison root: {comparison_root_for_map}")
print(f"Using metadata file: {diagnostics_metadata_file.name}")
print(f"Using locked session_id: {map_session_id}")
print(f"Located metadata rows for year {target_date.year}: {located_n_for_year}")
display(Image(filename=str(daily_map_output_path)))

## Interactive daily rainfall map

`scripts/diagnostics/plot_daily_rainfall_interactive.py` draws the same map as an
interactive [Plotly](https://plotly.com/python/) figure. Hovering over a point
shows the **specifier** &mdash; the ensemble transcription's source file name
&mdash; that the data comes from, along with the station name and consensus
rainfall value (in inches).

**Clicking a station copies its specifier to the clipboard** and shows it in a
read-only box above the map (with a *Copy* button and a manual-select fallback).
The cell below renders the figure as HTML so this works inline in the notebook.

All located stations for the selected year are shown; those with no value for
the day are displayed at zero (dry/pale end of the colour scale).

The figure is also written to a self-contained HTML file that can be opened in a
browser without a running kernel.

In [7]:
# Demonstrate the interactive daily-rainfall map, rendered inline.

import importlib.util
import os
from datetime import date
from pathlib import Path

import duckdb
from IPython.display import HTML, display

daily_map_interactive_script_path = (
    repo_root / "scripts" / "diagnostics" / "plot_daily_rainfall_interactive.py"
)

spec_daily_map_interactive = importlib.util.spec_from_file_location(
    "daily_map_interactive_plot", daily_map_interactive_script_path
)
daily_map_interactive_plot = importlib.util.module_from_spec(spec_daily_map_interactive)
spec_daily_map_interactive.loader.exec_module(daily_map_interactive_plot)

def _located_count_for_year(metadata_file: Path, target_year: int) -> int:
    conn = duckdb.connect()
    try:
        n = conn.execute(
            f"""
            SELECT COUNT(*)
            FROM read_parquet('{metadata_file}')
            WHERE matched_year = ?
              AND matched_latitude IS NOT NULL
              AND matched_longitude IS NOT NULL
            """
            , [target_year]
        ).fetchone()[0]
    finally:
        conn.close()
    return int(n)

target_date = date(1891, 11, 13)  # Change to any YYYY-MM-DD with matched records.
comparison_root_for_map = diagnostics_comparison_root
map_session_id = int(diagnostics_session_id)
located_n_for_year = _located_count_for_year(
    diagnostics_metadata_file, target_date.year
)
if located_n_for_year <= 0:
    raise SystemExit(
        f"No located ensemble records found for {target_date.isoformat()} "
        f"in locked metadata file {diagnostics_metadata_file.name}."
    )

daily_map_interactive_output_path = Path(
    f"{os.getenv('PDIR')}/diagnostics/daily_map_{target_date.isoformat()}.html"
)

interactive_fig = daily_map_interactive_plot.build_figure(
    target_date=target_date,
    ensemble_dataset_root=diagnostics_ensemble_root,
    comparison_root=comparison_root_for_map,
    session_id=map_session_id,
    output_path=daily_map_interactive_output_path,
)

print(f"Date: {target_date.isoformat()}")
print(f"Using comparison root: {comparison_root_for_map}")
print(f"Using metadata file: {diagnostics_metadata_file.name}")
print(f"Using locked session_id: {map_session_id}")
print(f"Located metadata rows for year {target_date.year}: {located_n_for_year}")

# Render only the custom HTML version so the station-selection/copy UI appears.
display(HTML(daily_map_interactive_plot.inline_html(interactive_fig)))

Date: 1891-11-13
Using comparison root: /data/scratch/philip.brohan/ADRQ/monthly_similarity_parquet
Using metadata file: session_000002.parquet
Using locked session_id: 2
Located metadata rows for year 1891: 4070


## Ensemble files matched per RR reference station (interactive)

`scripts/diagnostics/plot_rr_match_counts_interactive.py` maps, for a single
Rainfall-Rescue **reference year**, every RR station-year in that year, coloured
by **how many ensemble files are an exact match to it**.

"Exact match" uses the same rule as the metadata-assignment algorithm above
(the cell that runs `assign_ensemble_metadata_parquet`): an ensemble file's
**rank-1** similarity match with `exact_agreement_count >= 9`. Rank-1 matches
below that threshold — which the assignment step treats as approximate/centroid
matches — are **not** counted here.

This is a matching-health diagnostic. If the matcher is behaving, the exact
matches are spread thinly across many stations (most stations pale, counts near
1). An *attractor* year instead shows a few very bright stations that have soaked
up a disproportionate share of the ensemble files — a warning that those
station-years are acting as generic best-matches and inflating that year's
station count downstream (e.g. in the SEF export).

Grey markers are RR stations with no exact match; the colour scale (sqrt of the
count) covers stations with at least one exact match. **Clicking a station**
shows its RR label (location, station number, year, coordinates) and lists every
ensemble specifier exactly matched to that station, with a *Copy* button.


In [ ]:
# Demonstrate the RR-match-count map for one reference year, rendered inline.

import importlib.util
import os
from pathlib import Path

from IPython.display import HTML, display

rr_match_script_path = (
    repo_root / "scripts" / "diagnostics" / "plot_rr_match_counts_interactive.py"
)

spec_rr_match = importlib.util.spec_from_file_location(
    "rr_match_counts_plot", rr_match_script_path
)
rr_match_plot = importlib.util.module_from_spec(spec_rr_match)
spec_rr_match.loader.exec_module(rr_match_plot)

reference_year = 1891  # Change to any RR reference year to inspect its matches.
rr_match_output_path = Path(
    f"{os.getenv('PDIR')}/diagnostics/rr_match_counts_{reference_year}.html"
)

# Similarity and metadata sessions are different ids in the combined pipeline.
similarity_session_id = int(diagnostics_similarity_session_id)
metadata_session_id = int(diagnostics_session_id)

rr_match_fig = rr_match_plot.build_figure(
    year=reference_year,
    comparison_root=diagnostics_comparison_root,
    session_id=similarity_session_id,
    metadata_session_id=metadata_session_id,
    output_path=rr_match_output_path,
)

print(f"Comparison root: {diagnostics_comparison_root}")
print(f"Similarity session (locked): {similarity_session_id}")
print(f"Metadata session (locked):   {metadata_session_id}")
print(f"Reference year: {reference_year}")
# Render as HTML (not fig.show()) so the click-to-list JavaScript runs inline:
# click a station to see its RR label and the ensemble files matching it.
display(HTML(rr_match_plot.inline_html(rr_match_fig)))

## Sample example specifiers by fitting category

To eyeball how each kind of fitting result looks, the cell below draws a random
set of ensemble specifiers from the latest combined `ensemble_metadata` session
for a chosen category:

- **`exact_data`** - exact match against the DATA Rainfall-Rescue records
  (`match_type = 'exact'`; has coordinates).
- **`exact_allsheets`** - exact match against an ALLSHEETS source sheet
  (`match_type = 'exact_allsheets'`; coordinates present only where the name was
  found in `LeftOverSites.csv`).
- **`approximate`** - top-3 centroid match (`match_type = 'approximate'`).
- **`none`** - no match (all metadata NULL).

Set `sample_category` and `n_samples`, then paste any returned specifier into the
single-transcription diagnostic cell above to inspect that case. Set
`random_seed` to a float in `[-1, 1]` for a repeatable draw.


In [ ]:
# Sample example specifiers for a chosen fitting category.
#
# Pick a category and a sample size; the cell draws that many ensemble records
# at random from the locked ensemble_metadata session and prints their
# specifiers (file-name stems). Paste a specifier into the single-transcription
# diagnostic cell above to inspect that case.

import duckdb
from pathlib import Path

# --- choose here -----------------------------------------------------------
sample_category = "none"  # exact_data | exact_allsheets | approximate | none
n_samples = 25
random_seed = None  # float in [-1, 1] for a reproducible draw, or None for fresh
# ---------------------------------------------------------------------------

_CATEGORY_FILTER = {
    "exact_data": "match_type = 'exact'",
    "exact_allsheets": "match_type = 'exact_allsheets'",
    "approximate": "match_type = 'approximate'",
    "none": "match_type IS NULL",
}
if sample_category not in _CATEGORY_FILTER:
    raise ValueError(
        f"Unknown category {sample_category!r}; choose from {list(_CATEGORY_FILTER)}"
    )

# Locked metadata file from the provenance cell.
_meta_file = diagnostics_metadata_file
if not _meta_file.exists():
    raise FileNotFoundError(f"Expected metadata file not found: {_meta_file}")
_where = _CATEGORY_FILTER[sample_category]

_con = duckdb.connect()
try:
    if random_seed is not None:
        _con.execute("SELECT setseed(?)", [float(random_seed)])
    _total = _con.execute(
        f"SELECT COUNT(*) FROM read_parquet('{_meta_file}') WHERE {_where}"
    ).fetchone()[0]
    _rows = _con.execute(
        f"""
        SELECT file_name, matched_location_name, matched_year,
               matched_latitude, matched_longitude
        FROM read_parquet('{_meta_file}')
        WHERE {_where}
        ORDER BY random()
        LIMIT {int(n_samples)}
        """
    ).fetchall()
finally:
    _con.close()

sample_specifiers = [Path(r[0]).stem for r in _rows]

print(f"Comparison root : {diagnostics_comparison_root}")
print(f"Metadata file   : {_meta_file.name}")
print(f"Session id      : {int(diagnostics_session_id)}")
print(f"Category        : {sample_category}  ({_total:,} records total)")
print(f"Random sample of {len(_rows)}:\n")
for _r, _spec in zip(_rows, sample_specifiers):
    _loc = _r[1] or "n/a"
    _year = _r[2] if _r[2] is not None else "n/a"
    _coord = (
        f"lat={_r[3]:.2f} lon={_r[4]:.2f}"
        if _r[3] is not None and _r[4] is not None
        else "no coords"
    )
    print(f"  {_spec}")
    print(f"      loc={_loc}  year={_year}  {_coord}")